# Odometry Exercise 5 — plotting gyroscope bias and heading

Load a timestamped gyroscope log, inspect the raw axes and stationary rate, estimate bias from one chosen stationary trial, integrate separate records, and compare gyro and encoder turns with your independent physical-heading measurements.

This notebook is an introduction to plotting odometry data. Start with the supplied synthetic example so that you can see what each cell produces. The example is not evidence about your robot. When you are ready, change the settings in the **Use your own data** cell and run the notebook again.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(2026)


## 1. Use the example or your own data

Leave `USE_EXAMPLE_DATA` set to `True` on your first run. To use your measurements, upload the CSV received from the robot, set `USE_EXAMPLE_DATA = False`, and enter its filename.

Also enter the trial used for the axis check, the one stationary trial used to estimate bias, and the separate stationary trials used to evaluate it. These choices should match the labels you assigned before examining the results. This is the main cell you need to edit.


In [ ]:
USE_EXAMPLE_DATA = True
CSV_FILENAME = "odometry_exercise05_gyro_samples.csv"

AXIS_CHECK_TRIAL_NUM = 0
BIAS_TRIAL_NUM = 1
STATIONARY_EVALUATION_TRIALS = [2, 3]

print("Using:", "synthetic example" if USE_EXAMPLE_DATA else CSV_FILENAME)


## 2. Create the synthetic example

Run this cell when using the example. You do not need to understand or edit the generation code. It creates one axis check, three stationary records, and repeated clockwise and anticlockwise turns. The generated values are not evidence about your robot and are not results you should expect to reproduce.


In [ ]:
rng = np.random.default_rng(2026)
example_rows = []
sequence = 0
sample_period_s = 0.01
gyro_scale_dps_per_count = 0.00875

axis_times_s = np.arange(0, 4.01, sample_period_s)
axis_yaw_rate_dps = np.where(
    (axis_times_s >= 1.0) & (axis_times_s < 2.0),
    35.0,
    np.where(
        (axis_times_s >= 2.5) & (axis_times_s < 3.5),
        -30.0,
        0.0,
    ),
)
for elapsed_s, yaw_rate_dps in zip(axis_times_s, axis_yaw_rate_dps):
    sequence += 1
    measured_yaw_rate_dps = yaw_rate_dps + 0.18 + rng.normal(0, 0.05)
    example_rows.append({
        "test_name": "gyro",
        "trial_num": 0,
        "condition": "axis_check",
        "controller_time_ms": round(elapsed_s * 1000),
        "imu_time_us": round(elapsed_s * 1_000_000),
        "imu_sequence": sequence,
        "gyro_x_raw": round(rng.normal(5, 3)),
        "gyro_y_raw": round(rng.normal(-4, 3)),
        "gyro_z_raw": round(measured_yaw_rate_dps / gyro_scale_dps_per_count),
        "yaw_rate_dps": measured_yaw_rate_dps,
        "left_encoder_count": 0,
        "right_encoder_count": 0,
        "odometry_theta_rad": 0.0,
    })

stationary_conditions = [
    (1, 0.18),
    (2, 0.21),
    (3, 0.27),
]
for trial_num, stationary_bias_dps in stationary_conditions:
    stationary_times_s = np.arange(0, 20.01, sample_period_s)
    trial_start_ms = trial_num * 100_000
    for elapsed_s in stationary_times_s:
        sequence += 1
        measured_yaw_rate_dps = (
            stationary_bias_dps + rng.normal(0, 0.05)
        )
        example_rows.append({
            "test_name": "gyro",
            "trial_num": trial_num,
            "condition": "stationary",
            "controller_time_ms": trial_start_ms + round(elapsed_s * 1000),
            "imu_time_us": (
                trial_start_ms * 1000 + round(elapsed_s * 1_000_000)
            ),
            "imu_sequence": sequence,
            "gyro_x_raw": round(rng.normal(5, 3)),
            "gyro_y_raw": round(rng.normal(-4, 3)),
            "gyro_z_raw": round(
                measured_yaw_rate_dps / gyro_scale_dps_per_count
            ),
            "yaw_rate_dps": measured_yaw_rate_dps,
            "left_encoder_count": 0,
            "right_encoder_count": 0,
            "odometry_theta_rad": 0.0,
        })

turn_conditions = [
    (10, "clockwise", -88.0, -91.0),
    (11, "clockwise", -91.0, -94.0),
    (12, "clockwise", -89.0, -92.0),
    (13, "anticlockwise", 89.5, 87.0),
    (14, "anticlockwise", 92.0, 89.0),
    (15, "anticlockwise", 90.0, 88.0),
    (16, "clockwise", -134.0, -139.0),
    (17, "anticlockwise", 136.0, 132.0),
]
for trial_num, direction, physical_angle_deg, odometry_angle_deg in turn_conditions:
    turn_times_s = np.arange(0, 6.01, sample_period_s)
    turn_shape = np.zeros_like(turn_times_s)
    moving = (turn_times_s >= 1.0) & (turn_times_s <= 5.0)
    turn_shape[moving] = np.sin(
        np.pi * (turn_times_s[moving] - 1.0) / 4.0
    )
    true_rate_dps = (
        physical_angle_deg
        * turn_shape
        / (turn_shape.sum() * sample_period_s)
    )
    progress = np.cumsum(turn_shape)
    progress = progress / progress[-1]
    turn_sign = np.sign(physical_angle_deg)
    trial_start_ms = trial_num * 100_000

    for sample_num, elapsed_s in enumerate(turn_times_s):
        sequence += 1
        measured_yaw_rate_dps = (
            true_rate_dps[sample_num] + 0.21 + rng.normal(0, 0.08)
        )
        wheel_count = abs(physical_angle_deg) * 3.0 * progress[sample_num]
        example_rows.append({
            "test_name": "gyro",
            "trial_num": trial_num,
            "condition": direction,
            "controller_time_ms": trial_start_ms + round(elapsed_s * 1000),
            "imu_time_us": (
                trial_start_ms * 1000 + round(elapsed_s * 1_000_000)
            ),
            "imu_sequence": sequence,
            "gyro_x_raw": round(rng.normal(5, 4)),
            "gyro_y_raw": round(rng.normal(-4, 4)),
            "gyro_z_raw": round(
                measured_yaw_rate_dps / gyro_scale_dps_per_count
            ),
            "yaw_rate_dps": measured_yaw_rate_dps,
            "left_encoder_count": round(-turn_sign * wheel_count),
            "right_encoder_count": round(turn_sign * wheel_count),
            "odometry_theta_rad": np.radians(
                odometry_angle_deg * progress[sample_num]
            ),
        })

example_data = pd.DataFrame(example_rows)


## 3. Load and preview the selected data

When `USE_EXAMPLE_DATA` is false, `pd.read_csv(...)` reads your file. The final line displays its first five rows. This is the point where your uploaded measurements enter the notebook.


In [ ]:
if USE_EXAMPLE_DATA:
    data = example_data.copy()
else:
    data = pd.read_csv(CSV_FILENAME)

data.head()


## 4. Check sequence numbers and sample intervals

Before integrating a rate, check that each trial contains fresh samples in time order. The table below reports the typical and largest interval and counts repeated or backwards sequence numbers. Investigate any repeat, backwards step or unexplained long interval; do not integrate across it.


In [ ]:
trial_columns = ["condition", "trial_num"]
data = data.sort_values(trial_columns + ["imu_time_us"]).copy()
data["imu_interval_s"] = (
    data.groupby(trial_columns)["imu_time_us"].diff() / 1_000_000
)
data["sequence_step"] = data.groupby(trial_columns)["imu_sequence"].diff()
data["repeated_sequence"] = data["sequence_step"] == 0
data["backwards_sequence"] = data["sequence_step"] < 0

timing_summary = (
    data.groupby(trial_columns, as_index=False)
    .agg(
        sample_count=("imu_sequence", "size"),
        median_interval_s=("imu_interval_s", "median"),
        largest_interval_s=("imu_interval_s", "max"),
        repeated_sequences=("repeated_sequence", "sum"),
        backwards_sequences=("backwards_sequence", "sum"),
    )
)
timing_summary


## 5. Calculate elapsed time and readable trial labels

The StampC3 timestamps do not normally begin at zero. Subtracting the first IMU timestamp in each trial makes the plots easier to compare. The original timestamp and interval columns are retained. A condition is included in each label so that trial numbers may be reused.


In [ ]:
data["elapsed_s"] = (
    data["imu_time_us"]
    - data.groupby(trial_columns)["imu_time_us"].transform("first")
) / 1_000_000
data["trial_label"] = (
    data["condition"].str.replace("_", " ").str.title()
    + " "
    + data["trial_num"].astype(str)
)

data[[
    "condition", "trial_num", "imu_time_us", "imu_interval_s", "elapsed_s"
]].head()


## 6. Plot the three raw axes from the axis check

This plot helps you connect the physical rotation you performed with the responding sensor channel. The notebook does not choose the yaw axis for you: use your observation of the robot and record the axis and sign you established.


In [ ]:
axis_check = data.loc[
    (data["condition"] == "axis_check")
    & (data["trial_num"] == AXIS_CHECK_TRIAL_NUM)
].copy()
axis_plot_data = axis_check.melt(
    id_vars=["elapsed_s"],
    value_vars=["gyro_x_raw", "gyro_y_raw", "gyro_z_raw"],
    var_name="gyro_axis",
    value_name="raw_reading",
)

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.lineplot(
    data=axis_plot_data,
    x="elapsed_s",
    y="raw_reading",
    hue="gyro_axis",
    estimator=None,
    ax=ax,
)
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set(
    title="Raw gyroscope axes during the physical axis check",
    xlabel="Elapsed time (s)",
    ylabel="Raw gyroscope reading (counts)",
)
plt.show()


## 7. Plot the stationary yaw-rate records

These are direct converted rate measurements. Compare the typical level within each record with the variation between successive samples. A non-zero rate is a candidate bias; drift is what appears after that rate is integrated.


In [ ]:
stationary_data = data.loc[
    data["condition"] == "stationary"
].copy()

grid = sns.relplot(
    data=stationary_data,
    x="elapsed_s",
    y="yaw_rate_dps",
    col="trial_label",
    col_wrap=2,
    kind="line",
    estimator=None,
    height=3.2,
)
grid.set_axis_labels("Elapsed time (s)", "Yaw rate (degrees/s)")
grid.set_titles("{col_name}")
grid.figure.suptitle("Stationary gyroscope rate", y=1.03)
for axis in grid.axes.flat:
    axis.axhline(0, color="black", linewidth=1, linestyle="--")
plt.show()


## 8. Estimate bias from the chosen calibration trial

Only `BIAS_TRIAL_NUM` is used here. Its mean rate becomes the fixed bias estimate used below. The standard deviation describes the short-term spread in this one record; it is not another bias.


In [ ]:
bias_readings = stationary_data.loc[
    stationary_data["trial_num"] == BIAS_TRIAL_NUM,
    "yaw_rate_dps",
]
bias_estimate_dps = bias_readings.mean()
bias_spread_dps = bias_readings.std()

bias_summary = pd.DataFrame({
    "bias_trial_num": [BIAS_TRIAL_NUM],
    "mean_bias_dps": [bias_estimate_dps],
    "standard_deviation_dps": [bias_spread_dps],
    "sample_count": [len(bias_readings)],
})
bias_summary


## 9. Integrate the separate stationary evaluation trials

For each row, the timestamp difference gives the duration for which the previous rate is applied. The cumulative sum produces an angle. Both the uncorrected and bias-corrected headings start at zero. The bias is not recalculated from these evaluation trials.


In [ ]:
stationary_evaluation = stationary_data.loc[
    stationary_data["trial_num"].isin(STATIONARY_EVALUATION_TRIALS)
].copy()
stationary_evaluation = stationary_evaluation.sort_values(
    ["trial_num", "imu_time_us"]
)
stationary_evaluation["delta_time_s"] = (
    stationary_evaluation.groupby("trial_num")["imu_time_us"].diff()
    / 1_000_000
).fillna(0)
stationary_evaluation["previous_yaw_rate_dps"] = (
    stationary_evaluation.groupby("trial_num")["yaw_rate_dps"].shift(1)
)
stationary_evaluation["uncorrected_angle_step_deg"] = (
    stationary_evaluation["previous_yaw_rate_dps"]
    * stationary_evaluation["delta_time_s"]
).fillna(0)
stationary_evaluation["corrected_angle_step_deg"] = (
    (stationary_evaluation["previous_yaw_rate_dps"] - bias_estimate_dps)
    * stationary_evaluation["delta_time_s"]
).fillna(0)
stationary_evaluation["uncorrected_heading_deg"] = (
    stationary_evaluation.groupby("trial_num")[
        "uncorrected_angle_step_deg"
    ].cumsum()
)
stationary_evaluation["corrected_heading_deg"] = (
    stationary_evaluation.groupby("trial_num")[
        "corrected_angle_step_deg"
    ].cumsum()
)

stationary_heading_plot = stationary_evaluation.melt(
    id_vars=["trial_num", "trial_label", "elapsed_s"],
    value_vars=[
        "uncorrected_heading_deg",
        "corrected_heading_deg",
    ],
    var_name="calculation",
    value_name="heading_deg",
)
stationary_heading_plot["calculation"] = (
    stationary_heading_plot["calculation"]
    .str.replace("_heading_deg", "", regex=False)
    .str.replace("_", " ", regex=False)
    .str.title()
)

grid = sns.relplot(
    data=stationary_heading_plot,
    x="elapsed_s",
    y="heading_deg",
    hue="calculation",
    col="trial_label",
    kind="line",
    estimator=None,
    height=3.5,
)
grid.set_axis_labels("Elapsed time (s)", "Integrated heading (degrees)")
grid.set_titles("{col_name}")
grid.figure.suptitle("Heading drift in held-back stationary trials", y=1.04)
for axis in grid.axes.flat:
    axis.axhline(0, color="black", linewidth=1, linestyle="--")
plt.show()


## 10. Summarise final stationary drift

This table extracts the final integrated value from each evaluation record. Retain the time-series plots: the endpoint alone would hide settling or a changing rate.


In [ ]:
stationary_drift_summary = (
    stationary_evaluation.groupby("trial_num", as_index=False)
    .agg(
        duration_s=("elapsed_s", "last"),
        uncorrected_final_drift_deg=(
            "uncorrected_heading_deg",
            "last",
        ),
        corrected_final_drift_deg=("corrected_heading_deg", "last"),
    )
)
stationary_drift_summary


## 11. Integrate the turns and plot both onboard estimates

The gyroscope heading uses the fixed stationary bias. The encoder heading is converted from the local odometry radians. These lines are two onboard estimates; neither line is a measured physical trajectory.


In [ ]:
turn_data = data.loc[
    data["condition"].isin(["clockwise", "anticlockwise"])
].copy()
turn_columns = ["condition", "trial_num"]
turn_data = turn_data.sort_values(turn_columns + ["imu_time_us"])
turn_data["delta_time_s"] = (
    turn_data.groupby(turn_columns)["imu_time_us"].diff() / 1_000_000
).fillna(0)
turn_data["previous_yaw_rate_dps"] = (
    turn_data.groupby(turn_columns)["yaw_rate_dps"].shift(1)
)
turn_data["gyro_angle_step_deg"] = (
    (turn_data["previous_yaw_rate_dps"] - bias_estimate_dps)
    * turn_data["delta_time_s"]
).fillna(0)
turn_data["gyro_heading_change_deg"] = (
    turn_data.groupby(turn_columns)["gyro_angle_step_deg"].cumsum()
)
turn_data["encoder_heading_change_deg"] = np.degrees(
    turn_data["odometry_theta_rad"]
    - turn_data.groupby(turn_columns)[
        "odometry_theta_rad"
    ].transform("first")
)

turn_heading_plot = turn_data.melt(
    id_vars=[
        "trial_num",
        "trial_label",
        "condition",
        "elapsed_s",
    ],
    value_vars=[
        "gyro_heading_change_deg",
        "encoder_heading_change_deg",
    ],
    var_name="estimate",
    value_name="heading_change_deg",
)
turn_heading_plot["estimate"] = (
    turn_heading_plot["estimate"]
    .str.replace("_heading_change_deg", "", regex=False)
    .str.title()
)

grid = sns.relplot(
    data=turn_heading_plot,
    x="elapsed_s",
    y="heading_change_deg",
    hue="estimate",
    col="trial_label",
    col_wrap=2,
    kind="line",
    estimator=None,
    height=3,
)
grid.set_axis_labels("Elapsed time (s)", "Estimated heading change (degrees)")
grid.set_titles("{col_name}")
grid.figure.suptitle("Gyro and encoder heading estimates", y=1.02)
for axis in grid.axes.flat:
    axis.axhline(0, color="black", linewidth=0.8, linestyle="--")
plt.show()


## 12. Extract one endpoint row per turn

The complete time series remains in `turn_data`. This smaller summary takes the final gyro and encoder heading change from each trial so that they can be joined to the physical start and final headings.


In [ ]:
turn_summary = (
    turn_data.groupby(["condition", "trial_num"], as_index=False)
    .agg(
        gyro_heading_change_deg=("gyro_heading_change_deg", "last"),
        encoder_heading_change_deg=(
            "encoder_heading_change_deg",
            "last",
        ),
    )
    .rename(columns={"condition": "direction"})
)
turn_summary


## 13. Enter your independently measured headings

The robot log cannot recover the headings you measured from the printed reference. When using your own data, replace the example trial numbers and every `np.nan` below. Use degrees and the same positive anticlockwise convention as the exercise. Do not substitute the commanded turn for a physical measurement.


In [ ]:
if USE_EXAMPLE_DATA:
    physical_measurements = pd.DataFrame({
        "direction": ["clockwise"] * 3 + ["anticlockwise"] * 3 + ["clockwise", "anticlockwise"],
        "trial_num": [10, 11, 12, 13, 14, 15, 16, 17],
        "measured_start_heading_deg": [0.5, -0.5, 0.0, 0.0, 0.5, -0.5, 0.0, 0.5],
        "measured_final_heading_deg": [-87.5, -91.5, -89.0, 89.5, 92.5, 89.5, -134.0, 136.5],
    })
else:
    physical_measurements = pd.DataFrame({
        "direction": ["clockwise", "anticlockwise"],
        "trial_num": [1, 1],
        "measured_start_heading_deg": [np.nan, np.nan],
        "measured_final_heading_deg": [np.nan, np.nan],
    })

physical_measurements


## 14. Calculate physical heading changes and residuals

The measured change is final physical heading minus initial physical heading. Each residual follows the exercise convention: measured physical change minus the corresponding onboard estimate.


In [ ]:
comparison = turn_summary.merge(
    physical_measurements,
    on=["direction", "trial_num"],
)
comparison["measured_heading_change_deg"] = (
    comparison["measured_final_heading_deg"]
    - comparison["measured_start_heading_deg"]
)
comparison["gyro_residual_deg"] = (
    comparison["measured_heading_change_deg"]
    - comparison["gyro_heading_change_deg"]
)
comparison["encoder_residual_deg"] = (
    comparison["measured_heading_change_deg"]
    - comparison["encoder_heading_change_deg"]
)

comparison


## 15. Compare measured and estimated turn endpoints

Points on the dashed identity line indicate agreement with the independently measured heading change. The commanded angle is not used as the x-axis measurement.


In [ ]:
endpoint_comparison = comparison.melt(
    id_vars=[
        "trial_num",
        "direction",
        "measured_heading_change_deg",
    ],
    value_vars=[
        "gyro_heading_change_deg",
        "encoder_heading_change_deg",
    ],
    var_name="estimate",
    value_name="estimated_heading_change_deg",
)
endpoint_comparison["estimate"] = (
    endpoint_comparison["estimate"]
    .str.replace("_heading_change_deg", "", regex=False)
    .str.title()
)

fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(
    data=endpoint_comparison,
    x="measured_heading_change_deg",
    y="estimated_heading_change_deg",
    hue="estimate",
    style="direction",
    s=100,
    ax=ax,
)
ax.axline((0, 0), slope=1, color="black", linestyle="--")
ax.set(
    title="Physical, gyro and encoder heading changes",
    xlabel="Measured physical heading change (degrees)",
    ylabel="Estimated heading change (degrees)",
)
plt.show()


## 16. Plot the endpoint residuals

Zero means that an onboard estimate agrees with the independently measured heading change for that trial. Keep the two estimate sources visible rather than combining them into one score.


In [ ]:
residual_plot_data = comparison.melt(
    id_vars=["trial_num", "direction"],
    value_vars=["gyro_residual_deg", "encoder_residual_deg"],
    var_name="estimate",
    value_name="residual_deg",
)
residual_plot_data["estimate"] = (
    residual_plot_data["estimate"]
    .str.replace("_residual_deg", "", regex=False)
    .str.title()
)
residual_plot_data["trial_label"] = (
    residual_plot_data["direction"].str.title()
    + " "
    + residual_plot_data["trial_num"].astype(str)
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(
    data=residual_plot_data,
    x="trial_label",
    y="residual_deg",
    hue="estimate",
    style="direction",
    s=100,
    ax=ax,
)
ax.axhline(0, color="black", linestyle="--")
ax.set(
    title="Physical heading change minus onboard estimate",
    xlabel="Trial",
    ylabel="Endpoint residual (degrees)",
)
plt.show()


## What to notice

- Which raw axis responded clearly when you rotated the complete chassis, and did its sign match your heading convention?
- Is the stationary rate centred on zero, and does its centre appear the same in each stationary trial?
- Does subtracting the calibration bias reduce drift in the separate stationary records?
- Do gyro or encoder endpoint residuals change consistently with turn direction or turn size?

Save the notebook with your plots. The gyro and encoder curves are onboard estimates. Only your independently measured start and final headings assess the physical turn, and those endpoint measurements do not establish the path between them.
